# CatBoost Model Development

This notebook is the first tabular model-development workspace for hourly DK1/DK2 day-ahead price forecasts.

It deliberately follows the baseline notebook's evaluation shape:

- read the model-ready panel only
- use rolling-origin backtests
- predict an explicit Danish delivery-day horizon
- keep feature construction leakage-safe
- compare against the local baselines before trusting a more flexible model

CatBoost is an optional modeling dependency. The notebook is still useful without it: it builds the feature matrix, checks feature availability, runs the baseline comparator, and shows exactly where CatBoost training will run once installed.

```bash
pip install "catboost>=1.2"
```

<div style="color:#b42318; border-left:4px solid #b42318; padding:0.35rem 0 0.35rem 0.8rem; margin:0.75rem 0;"><strong>Interpretation convention.</strong> Red callouts are modeling interpretation, judgement, or next-step recommendations. Tables, charts, and code cells are the legwork; red text is my read of what the legwork suggests.</div>


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid", context="notebook")
except ImportError:
    sns = None
    plt.style.use("seaborn-v0_8-whitegrid")

try:
    import catboost  # noqa: F401
    HAS_CATBOOST = True
except ImportError:
    HAS_CATBOOST = False

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.labelsize"] = 11


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not find project root with pyproject.toml and src/")


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from dkenergy_forecast.backtesting.horizons import make_daily_origins, make_danish_delivery_day_horizon
from dkenergy_forecast.backtesting.rolling_origin import rolling_origin_backtest
from dkenergy_forecast.evaluation.point_metrics import bias, mae, rmse
from dkenergy_forecast.evaluation.probabilistic_metrics import average_interval_width, interval_coverage, pinball_loss
from dkenergy_forecast.evaluation.value_metrics import cheapest_k_hit_rate
from dkenergy_forecast.features.price_features import (
    PriceFeatureConfig,
    build_price_feature_frame,
    build_training_matrix as build_feature_training_matrix,
)
from dkenergy_forecast.io import load_price_panel
from dkenergy_forecast.models.baselines import LagNaive, SeasonalRollingMedian
from dkenergy_forecast.models.catboost_quantile import CatBoostQuantileModel
from dkenergy_forecast.types import add_copenhagen_calendar

PANEL_PATH = PROJECT_ROOT / "data" / "model_ready" / "price_panel_hourly_v1.parquet"
QA_PATH = PROJECT_ROOT / "data" / "model_ready" / "price_panel_hourly_v1.qa.json"

FORECAST_AT_HOUR_UTC = 10
DEVELOPMENT_BACKTEST_DAYS = 21
CATBOOST_MAX_ORIGINS = 8
TRAINING_ORIGINS_PER_MODEL = 70
CATBOOST_ITERATIONS = 250
QUANTILES = {"q10": 0.10, "q50": 0.50, "q90": 0.90}
K_CHEAPEST_HOURS = 6

INTERPRETATION_STYLE = (
    "color:#b42318; border-left:4px solid #b42318; "
    "padding:0.35rem 0 0.35rem 0.8rem; margin:0.75rem 0;"
)


def interpretation(text: str) -> None:
    display(Markdown(f'<div style="{INTERPRETATION_STYLE}"><strong>Interpretation.</strong> {text}</div>'))


print(f"Project root: {PROJECT_ROOT}")
print(f"Panel path: {PANEL_PATH}")
print(f"CatBoost installed: {HAS_CATBOOST}")


## 1. Load the panel

CatBoost should not change the data contract. We still consume only the model-ready panel and, when available, its QA artifact. If the panel is missing, this notebook creates a synthetic teaching panel so the feature-building path remains inspectable.


In [ ]:
def make_synthetic_price_panel(
    start: str = "2024-01-01T00:00:00Z",
    end: str = "2024-05-15T00:00:00Z",
) -> pd.DataFrame:
    rng = np.random.default_rng(7)
    timestamps = pd.date_range(start=start, end=end, freq="h", inclusive="left", tz="UTC")
    rows = []
    for area, area_offset, noise_scale in [("DK1", 0.0, 18.0), ("DK2", 16.0, 20.0)]:
        local = timestamps.tz_convert("Europe/Copenhagen")
        hour = local.hour.to_numpy()
        dow = local.dayofweek.to_numpy()
        day_index = np.arange(len(timestamps)) / 24
        y = (
            360
            + area_offset
            + 65 * np.exp(-0.5 * ((hour - 8) / 2.2) ** 2)
            + 85 * np.exp(-0.5 * ((hour - 19) / 2.8) ** 2)
            - np.where((hour <= 4) | (hour >= 23), 40, 0)
            - np.where(dow >= 5, 35, 0)
            + 30 * np.sin(2 * np.pi * day_index / 28)
            + rng.normal(0, noise_scale, len(timestamps))
        )
        frame = pd.DataFrame(
            {
                "unique_id": f"day_ahead_price_{area}",
                "ds_utc": timestamps,
                "area": area,
                "y": y.round(2),
                "dataset_version": "synthetic_catboost_demo_v1",
            }
        )
        rows.append(frame)
    panel = pd.concat(rows, ignore_index=True)
    panel = add_copenhagen_calendar(panel)
    panel["price_dkk_per_mwh"] = panel["y"]
    panel["price_eur_per_mwh"] = panel["y"] / 7.45
    return panel.sort_values(["area", "ds_utc"]).reset_index(drop=True)


qa = {}
if PANEL_PATH.exists():
    panel = load_price_panel(PANEL_PATH, QA_PATH if QA_PATH.exists() else None, require_final_historical=False)
    if QA_PATH.exists():
        qa = json.loads(QA_PATH.read_text(encoding="utf-8"))
    USING_SYNTHETIC = False
else:
    panel = make_synthetic_price_panel()
    USING_SYNTHETIC = True

panel = panel.copy()
panel["ds_utc"] = pd.to_datetime(panel["ds_utc"], utc=True)
panel["ds_local"] = pd.to_datetime(panel["ds_local"], utc=True).dt.tz_convert("Europe/Copenhagen")
panel = panel.sort_values(["area", "ds_utc"]).reset_index(drop=True)

display(Markdown(f"Loaded **{len(panel):,}** rows. Synthetic fallback: **{USING_SYNTHETIC}**."))
display(panel.head())

if USING_SYNTHETIC:
    interpretation("This is a synthetic run. Use it to inspect mechanics only; install/build the real panel before making modeling decisions.")
else:
    interpretation(
        f"Real panel loaded from {panel['ds_utc'].min()} to {panel['ds_utc'].max()}; QA status is `{qa.get('artifact_status', 'not supplied')}`."
    )


## 2. Choose rolling origins

To keep this first CatBoost notebook interactive, we use a short development window and cap CatBoost training to the most recent origins. The origin and horizon definition is intentionally identical to the baseline notebook.


In [ ]:
def choose_development_origins(panel: pd.DataFrame, days: int, at_hour_utc: int) -> pd.DataFrame:
    min_origin = (panel["ds_utc"].min() + pd.Timedelta(days=90)).normalize()
    max_origin = (panel["ds_utc"].max() - pd.Timedelta(days=2)).normalize()
    start = max(min_origin, max_origin - pd.Timedelta(days=days))
    end = max_origin + pd.Timedelta(days=1)
    origins = make_daily_origins(panel, start=start, end=end, at_hour_utc=at_hour_utc)

    valid = []
    for origin in origins["forecast_origin_utc"]:
        horizon = make_danish_delivery_day_horizon(panel, origin, days_ahead=1)
        if horizon["ds_utc"].min() >= panel["ds_utc"].min() and horizon["ds_utc"].max() <= panel["ds_utc"].max():
            valid.append(origin)
    return pd.DataFrame({"forecast_origin_utc": valid})


all_origins = choose_development_origins(panel, DEVELOPMENT_BACKTEST_DAYS, FORECAST_AT_HOUR_UTC)
catboost_origins = all_origins.tail(CATBOOST_MAX_ORIGINS).reset_index(drop=True)

display(
    pd.DataFrame(
        {
            "candidate_origins": [len(all_origins)],
            "catboost_origins": [len(catboost_origins)],
            "first_catboost_origin": [catboost_origins["forecast_origin_utc"].min()],
            "last_catboost_origin": [catboost_origins["forecast_origin_utc"].max()],
        }
    )
)

example_horizon = make_danish_delivery_day_horizon(panel, catboost_origins["forecast_origin_utc"].iloc[0], days_ahead=1)
display(example_horizon.groupby(["area", "local_date"]).size().rename("rows").reset_index())


## 3. Leakage-safe feature construction

The important design choice is not CatBoost itself; it is the feature table.

For a given forecast origin, every future feature must be known at that origin. This notebook therefore builds features from:

- known calendar fields for the delivery timestamp
- historical lags whose source timestamp is before the origin
- rolling and seasonal summaries computed only from rows before the origin
- historical DK1/DK2 spread lags whose source timestamp is before the origin

The training matrix is built from previous rolling origins using the same feature function. That makes the training rows resemble the forecast rows instead of silently using richer hindsight features.


In [ ]:
FEATURE_CONFIG = PriceFeatureConfig()
FEATURE_COLUMNS = FEATURE_CONFIG.feature_columns
CAT_FEATURES = list(FEATURE_CONFIG.categorical_features)


def build_origin_feature_frame(panel: pd.DataFrame, origin: pd.Timestamp, *, include_target: bool) -> pd.DataFrame:
    origin = pd.Timestamp(origin).tz_convert("UTC")
    future = make_danish_delivery_day_horizon(panel, origin, days_ahead=1)
    return build_price_feature_frame(
        panel,
        future,
        forecast_origin_utc=origin,
        include_target=include_target,
        config=FEATURE_CONFIG,
    )


def build_training_matrix(panel: pd.DataFrame, origin: pd.Timestamp, *, training_origins: int) -> pd.DataFrame:
    return build_feature_training_matrix(
        panel,
        origin,
        training_origin_days=training_origins,
        at_hour_utc=FORECAST_AT_HOUR_UTC,
        config=FEATURE_CONFIG,
    )


example_origin = catboost_origins["forecast_origin_utc"].iloc[0]
example_features = build_origin_feature_frame(panel, example_origin, include_target=True)
example_train = build_training_matrix(panel, example_origin, training_origins=14)

metadata_columns = [
    "unique_id",
    "area",
    "ds_utc",
    "ds_local",
    "local_date",
    "forecast_origin_utc",
    "horizon",
    "y",
    "dataset_version",
]
display(example_features[[column for column in metadata_columns if column in example_features.columns]].head())
display(
    pd.DataFrame(
        {
            "feature": FEATURE_COLUMNS,
            "missing_share_forecast_example": [example_features[column].isna().mean() for column in FEATURE_COLUMNS],
            "missing_share_training_example": [example_train[column].isna().mean() for column in FEATURE_COLUMNS],
        }
    )
)

interpretation(
    "The notebook now uses the same packaged feature builder as the scripted CatBoost run. That keeps the didactic exploration honest: feature metadata, leakage cutoffs, and production backtest behavior come from one implementation rather than two near-copies."
)


## 4. Baseline comparator on the same origins

Before fitting CatBoost, run the local baselines on the exact same origins. This guards against the classic notebook trap: getting excited about a flexible model before asking whether a transparent rule already does most of the work.


In [ ]:
baseline_specs = {
    "same_hour_last_week": lambda: LagNaive(lag_hours=168),
    "rolling_median_local_hour_28d": lambda: SeasonalRollingMedian(
        lookback_days=28,
        seasonal_keys=("local_hour",),
        min_periods=7,
    ),
    "rolling_median_hour_weekend_56d": lambda: SeasonalRollingMedian(
        lookback_days=56,
        seasonal_keys=("local_hour", "is_weekend"),
        min_periods=4,
    ),
}


def delivery_day_horizon_builder(panel: pd.DataFrame, origin: pd.Timestamp) -> pd.DataFrame:
    return make_danish_delivery_day_horizon(panel, origin, days_ahead=1)


baseline_frames = []
for model_label, factory in baseline_specs.items():
    preds = rolling_origin_backtest(
        model_factory=factory,
        panel=panel,
        origins=catboost_origins,
        horizon_builder=delivery_day_horizon_builder,
        min_train_rows=24 * 60 * panel["area"].nunique(),
    )
    preds["model_label"] = model_label
    baseline_frames.append(preds)

baseline_predictions = pd.concat(baseline_frames, ignore_index=True)
baseline_predictions["error"] = baseline_predictions["y_pred"] - baseline_predictions["y"]
baseline_predictions["abs_error"] = baseline_predictions["error"].abs()

def point_summary(predictions: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for model_label, frame in predictions.groupby("model_label"):
        rows.append(
            {
                "model_label": model_label,
                "rows": len(frame),
                "evaluated_rows": int(frame["y_pred"].notna().sum()),
                "mae": mae(frame),
                "rmse": rmse(frame),
                "bias": bias(frame),
            }
        )
    return pd.DataFrame(rows).sort_values("mae")

baseline_summary = point_summary(baseline_predictions)
display(baseline_summary)

fig, ax = plt.subplots(figsize=(10, 4))
if sns is not None:
    sns.barplot(data=baseline_summary, x="model_label", y="mae", ax=ax)
else:
    ax.bar(baseline_summary["model_label"], baseline_summary["mae"])
ax.set_title("Baseline MAE on CatBoost development origins")
ax.set_xlabel("")
ax.set_ylabel("MAE, DKK/MWh")
ax.tick_params(axis="x", labelrotation=30)
plt.tight_layout()
plt.show()

best_baseline = baseline_summary.iloc[0]
interpretation(
    f"Current baseline to beat on this CatBoost slice: `{best_baseline['model_label']}` with MAE {best_baseline['mae']:,.1f}. CatBoost should be judged against this, not against a strawman."
)


## 5. Optional CatBoost quantile backtest

This cell trains one CatBoost model per quantile and per rolling origin. It is intentionally small: enough to test the workflow, not enough to claim a final model.

If CatBoost is not installed, the cell exits cleanly after showing the install command.


In [ ]:
catboost_predictions = pd.DataFrame()
catboost_models: list[CatBoostQuantileModel] = []
feature_importance = pd.DataFrame(columns=["model_label", "forecast_origin_utc", "quantile", "feature", "importance"])

if not HAS_CATBOOST:
    install_message = "\n".join(
        [
            "**CatBoost is not installed in this environment.** Install it with:",
            "",
            "```bash",
            'pip install "catboost>=1.2"',
            "```",
            "",
            "The feature matrix and baseline comparator above still ran, so the notebook is ready for the optional model step.",
        ]
    )
    display(Markdown(install_message))
    interpretation(
        "Skipping CatBoost training is intentional here, not a failure. The project keeps CatBoost optional so the core data and baseline library remain lightweight."
    )
else:
    def catboost_model_factory() -> CatBoostQuantileModel:
        model = CatBoostQuantileModel(
            feature_config=FEATURE_CONFIG,
            quantiles=QUANTILES,
            training_origin_days=TRAINING_ORIGINS_PER_MODEL,
            at_hour_utc=FORECAST_AT_HOUR_UTC,
            iterations=CATBOOST_ITERATIONS,
            depth=6,
            learning_rate=0.05,
            l2_leaf_reg=8.0,
            random_seed=42,
            verbose=False,
            model_version="notebook_v1",
        )
        catboost_models.append(model)
        return model

    catboost_predictions = rolling_origin_backtest(
        model_factory=catboost_model_factory,
        panel=panel,
        origins=catboost_origins,
        horizon_builder=delivery_day_horizon_builder,
        min_train_rows=24 * 60 * panel["area"].nunique(),
    )
    catboost_predictions["model_label"] = "catboost_quantile"
    catboost_predictions["error"] = catboost_predictions["y_pred"] - catboost_predictions["y"]
    catboost_predictions["abs_error"] = catboost_predictions["error"].abs()

    feature_frames = [model.feature_importance_frame() for model in catboost_models]
    feature_frames = [frame for frame in feature_frames if not frame.empty]
    if feature_frames:
        feature_importance = pd.concat(feature_frames, ignore_index=True)
        feature_importance["model_label"] = "catboost_quantile"
        feature_importance = feature_importance[
            ["model_label", "forecast_origin_utc", "quantile", "feature", "importance"]
        ]

    display(catboost_predictions.head())
    display(catboost_predictions["raw_quantile_crossing_rate"].describe())
    interpretation(
        "CatBoost now runs through the same model protocol as the baselines: train only on rows before the forecast origin, predict on a target-free future frame, then join actuals afterward. This is slower than a hand-rolled notebook loop, but it is much harder to accidentally leak or drift from the library contract."
    )


## 6. CatBoost versus baselines

If CatBoost ran, compare q50 accuracy, pinball loss, p10-p90 coverage, interval width, and cheapest-hour hit rate. The aim is not only lower MAE; probabilistic forecasts should also produce useful uncertainty bands.


In [ ]:
if catboost_predictions.empty:
    display(Markdown("CatBoost comparison skipped because no CatBoost predictions were produced."))
else:
    combined_point = pd.concat(
        [
            baseline_predictions[["model_label", "area", "forecast_origin_utc", "ds_utc", "ds_local", "local_date", "y", "y_pred"]],
            catboost_predictions[["model_label", "area", "forecast_origin_utc", "ds_utc", "ds_local", "local_date", "y", "y_pred"]],
        ],
        ignore_index=True,
    )
    point = point_summary(combined_point)
    display(point)

    probabilistic = pd.DataFrame(
        [
            {"metric": "pinball_q10", "value": pinball_loss(catboost_predictions, quantile=0.10)},
            {"metric": "pinball_q50", "value": pinball_loss(catboost_predictions, quantile=0.50)},
            {"metric": "pinball_q90", "value": pinball_loss(catboost_predictions, quantile=0.90)},
            {"metric": "p10_p90_coverage", "value": interval_coverage(catboost_predictions)},
            {"metric": "p10_p90_avg_width", "value": average_interval_width(catboost_predictions)},
        ]
    )
    display(probabilistic)

    value_frames = []
    for label, frame in combined_point.groupby("model_label"):
        value = cheapest_k_hit_rate(frame, k=K_CHEAPEST_HOURS)
        value["model_label"] = label
        value_frames.append(value)
    value_summary = (
        pd.concat(value_frames, ignore_index=True)
        .groupby("model_label")
        .agg(mean_hit_rate=("hit_rate", "mean"), evaluated_days=("hit_rate", "size"))
        .sort_values("mean_hit_rate", ascending=False)
        .reset_index()
    )
    display(value_summary)

    fig, ax = plt.subplots(figsize=(10, 4))
    if sns is not None:
        sns.barplot(data=point, x="model_label", y="mae", ax=ax)
    else:
        ax.bar(point["model_label"], point["mae"])
    ax.set_title("CatBoost q50 versus local baselines")
    ax.set_xlabel("")
    ax.set_ylabel("MAE, DKK/MWh")
    ax.tick_params(axis="x", labelrotation=30)
    plt.tight_layout()
    plt.show()

    catboost_mae = point.loc[point["model_label"] == "catboost_quantile", "mae"].iloc[0]
    best_baseline_mae = point.loc[point["model_label"] != "catboost_quantile", "mae"].min()
    if catboost_mae < best_baseline_mae:
        interpretation(
            f"CatBoost beats the best local baseline on q50 MAE by {best_baseline_mae - catboost_mae:,.1f} DKK/MWh on this development slice. Next step: rerun on a longer window before trusting it."
        )
    else:
        interpretation(
            f"CatBoost does not beat the best local baseline on this slice. Treat this as feature-development feedback rather than a reason to tune harder immediately."
        )


## 7. Inspect one CatBoost forecast

A quantile model should be inspected visually. We want to see whether q10-q90 bands expand on volatile days and whether the median forecast misses ramps systematically.


In [ ]:
if catboost_predictions.empty:
    display(Markdown("Forecast plot skipped because CatBoost did not run."))
else:
    example_origin = catboost_predictions["forecast_origin_utc"].drop_duplicates().sort_values().iloc[-1]
    example_area = "DK1" if "DK1" in catboost_predictions["area"].unique() else catboost_predictions["area"].iloc[0]
    example = catboost_predictions.loc[
        (catboost_predictions["forecast_origin_utc"] == example_origin)
        & (catboost_predictions["area"] == example_area)
    ].sort_values("ds_utc")

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.plot(example["ds_local"], example["y"], color="black", linewidth=2.2, label="actual")
    ax.plot(example["ds_local"], example["q50"], color="#1f77b4", linewidth=1.8, label="CatBoost q50")
    ax.fill_between(example["ds_local"], example["q10"], example["q90"], color="#1f77b4", alpha=0.18, label="q10-q90")
    ax.set_title(f"CatBoost quantile forecast, {example_area}, origin {example_origin.isoformat()}")
    ax.set_xlabel("Danish local delivery time")
    ax.set_ylabel("DKK/MWh")
    ax.legend(loc="best")
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()


## 8. Feature importance

Feature importance is not causality, but it is still a useful smoke test. If CatBoost mostly uses leakage-prone or unavailable features, the design is wrong. If it leans on calendar, historical lags, rolling summaries, and spread lags, the first feature table is at least plausible.


In [ ]:
if feature_importance.empty:
    display(Markdown("Feature importance skipped because CatBoost did not run."))
else:
    q50_importance = (
        feature_importance.loc[feature_importance["quantile"] == "q50"]
        .groupby("feature", as_index=False)["importance"]
        .mean()
        .sort_values("importance", ascending=False)
    )
    display(q50_importance.head(20))

    fig, ax = plt.subplots(figsize=(10, 5))
    top = q50_importance.head(15).sort_values("importance")
    ax.barh(top["feature"], top["importance"])
    ax.set_title("CatBoost q50 feature importance, averaged across development origins")
    ax.set_xlabel("Mean importance")
    plt.tight_layout()
    plt.show()

    interpretation(
        "Use this importance table as a debugging lens, not as causal explanation. If the model leans almost entirely on one lag or calendar column, inspect that error slice before adding more features."
    )


## 9. Modeling notes from this pass

- This notebook introduces feature tables, not a production CatBoost wrapper.
- Training rows are generated from prior rolling origins using the same feature builder as forecast rows.
- Missing lag features are allowed and visible; they represent forecast-origin availability, not data corruption.
- CatBoost remains optional. The core package still imports without it.
- Promote CatBoost only after it beats the local baselines on a longer backtest window and produces sensible interval coverage.

The next implementation step, if this notebook looks promising on the real panel, is a thin optional `CatBoostQuantileModel` adapter that reuses this feature-building logic without hiding it inside the model class.
